In [ ]:
import requests
import pandas as pd
import numpy as np
import mysql.connector



LIMIT = 5000

API_URL = #"insert  your api url here"

response = requests.get(API_URL)
data = response.json()

df = pd.DataFrame(data)

print("Records Loaded:", len(df))


#  DATE PROCESSING 

df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")

df["complaint_date"] = df["created_date"].dt.date
df["complaint_time"] = df["created_date"].dt.time

df["year"] = df["created_date"].dt.year
df["month"] = df["created_date"].dt.month
df["day"] = df["created_date"].dt.day
df["hour"] = df["created_date"].dt.hour
df["weekday_name"] = df["created_date"].dt.day_name()
df["weekend_flag"] = df["weekday_name"].isin(["Saturday", "Sunday"])


# CRIME CLASSIFICATION 

def classify_crime(row):
    text = str(row.get("complaint_type", "")).lower()

    if "assault" in text or "rape" in text or "harassment" in text:
        return "Violent Crime", row.get("complaint_type")
    elif "theft" in text or "burglary" in text or "robbery" in text:
        return "Property Crime", row.get("complaint_type")
    elif "noise" in text or "loud" in text:
        return "Public Nuisance", "Noise Complaint"
    elif "parking" in text:
        return "Traffic Offense", "Illegal Parking"
    else:
        return "Administrative Case", row.get("complaint_type")


df[["crime_category", "crime_type"]] = df.apply(
    lambda x: pd.Series(classify_crime(x)), axis=1
)


# OTHER FIELDS 

df["assigned_department"] = df["agency_name"].fillna("NYC Services")
df["response_time_minutes"] = np.random.randint(5, 180, size=len(df))


def priority_logic(x):
    if x == "Violent Crime":
        return "Critical"
    elif x == "Property Crime":
        return "High"
    elif x == "Public Nuisance":
        return "Medium"
    else:
        return "Low"


df["case_priority"] = df["crime_category"].apply(priority_logic)

df["arrest_made"] = np.random.choice([True, False], size=len(df))
df["number_of_arrests"] = df["arrest_made"].apply(
    lambda x: np.random.randint(1, 3) if x else 0
)

df["status"] = df["status"].fillna("Open")

df["closed_date"] = pd.to_datetime(df.get("closed_date"), errors="coerce").dt.date
df["closed_time"] = pd.to_datetime(df.get("closed_date"), errors="coerce").dt.time

df = df.fillna({
    "incident_address": "Unknown",
    "borough": "Unknown",
    "city": "New York",
    "incident_zip": "00000",
    "latitude": 0,
    "longitude": 0
})


#  FINAL COLUMN SELECTION 

df_final = df[[
    "unique_key",
    "complaint_date",
    "complaint_time",
    "crime_type",
    "crime_category",
    "descriptor",
    "borough",
    "incident_address",
    "city",
    "incident_zip",
    "latitude",
    "longitude",
    "assigned_department",
    "response_time_minutes",
    "case_priority",
    "arrest_made",
    "number_of_arrests",
    "status",
    "closed_date",
    "closed_time",
    "year",
    "month",
    "day",
    "hour",
    "weekday_name",
    "weekend_flag"
]]

df_final.columns = [
    "complaint_id",
    "complaint_date",
    "complaint_time",
    "crime_type",
    "crime_category",
    "description",
    "borough",
    "incident_address",
    "city",
    "zipcode",
    "latitude",
    "longitude",
    "assigned_department",
    "response_time_minutes",
    "case_priority",
    "arrest_made",
    "number_of_arrests",
    "status",
    "closed_date",
    "closed_time",
    "year",
    "month",
    "day",
    "hour",
    "weekday_name",
    "weekend_flag"
]

df_final = df_final.replace({np.nan: None})


# MYSQL INSERT 


# connect to mysql
conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="2711",
    database="smart_city",
    autocommit=False
)

cursor = conn.cursor()

# check database connection
cursor.execute("SELECT DATABASE();")
print("Connected DB:", cursor.fetchone())


# clear old data before insert
cursor.execute("TRUNCATE TABLE public_safety_nyc")
conn.commit()


insert_query = """
INSERT INTO public_safety_nyc (
complaint_id, complaint_date, complaint_time,
crime_type, crime_category, description,
borough, incident_address, city, zipcode,
latitude, longitude, assigned_department,
response_time_minutes, case_priority,
arrest_made, number_of_arrests, status,
closed_date, closed_time,
year, month, day, hour, weekday_name, weekend_flag
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""


# convert dataframe to tuples
data_tuples = [tuple(row) for row in df_final.to_numpy()]

cursor.executemany(insert_query, data_tuples)

conn.commit()


# check rows inserted
cursor.execute("SELECT COUNT(*) FROM public_safety_nyc")
print("Rows:", cursor.fetchone())


print("Smart City Public Safety records inserted successfully.")


cursor.close()
conn.close()

Records Loaded: 5000
Connected DB: ('smart_city',)
Rows: (5000,)
Smart City Public Safety records inserted successfully.
